# CP5 · Fer la prova final i explicar el resultat

**Objectiu:** comprovar el model en dades reservades i comunicar els seus errors i límits.

### 1. Deixar per escrit el model, els predictors i els paràmetres abans d’executar la prova.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---------- DECISIÓ CONGELADA (vinguda de CP3, apartat 5.4) ----------
# Model:        Ridge (regressió lineal regularitzada, amb NumPy)
# Paràmetre:    alpha = 300 (triat només amb la validació de 2025)
# Entrenament:  2022-2024  (2021 només serveix per a crear lag_12)
# Validació:    2025       (ja utilitzada per a triar; no es toca més)
# Prova final:  gener-juny 2026 (s'avalua UNA sola vegada)
# Predictors (es coneixen abans del mes):
#   numèrics:    marketing_budget, planned_discount, stock_units, list_price,
#                days_in_month, lag_1, lag_12
#   categòrics:  channel, product_line, region, campaign
#   derivats:    month_sin, month_cos, time_index i sinus/cosinus per producte
# Excloses (se saben després): revenue, website_sessions, marketing_spend,
#   avg_discount, closing_stock_units, stockout
# Les prediccions es limiten entre 0 i l'estoc inicial.
ALPHA = 300
KEY = ['channel', 'product_line', 'region']
NUM = ['marketing_budget', 'planned_discount', 'stock_units', 'list_price',
       'days_in_month', 'lag_1', 'lag_12']
CAT = KEY + ['campaign']

# ---------- Dades i particions ----------
sales = pd.read_csv(Path('..') / 'dades' / 'sales_history.csv')
sales['month'] = pd.to_datetime(sales['month'], format='%Y-%m')
sales = sales.sort_values(KEY + ['month']).reset_index(drop=True)

for lag in [1, 12]:
    sales[f'lag_{lag}'] = sales.groupby(KEY)['sales_units'].shift(lag)
usable = sales.dropna(subset=['lag_1', 'lag_12'])

train = usable[usable['month'] < '2025-01-01'].copy()
test = usable[usable['month'] >= '2026-01-01'].copy()
print('Entrenament:', len(train), 'files · Prova:', len(test), 'files')
print('Mesos de la prova:', test['month'].dt.strftime('%Y-%m').unique())

# ---------- Funcions (les mateixes que en CP3) ----------
def score(y, pred):
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    return {'MAE': np.mean(np.abs(y - pred)), 'WAPE_%': 100 * np.abs(y - pred).sum() / y.sum()}

def cap(frame, pred):
    return np.clip(pred, 0, frame['stock_units'].to_numpy())

def design(frame):
    x = frame[NUM].copy()
    month = frame['month'].dt.month
    x['month_sin'] = np.sin(2 * np.pi * month / 12)
    x['month_cos'] = np.cos(2 * np.pi * month / 12)
    x['time_index'] = (frame['month'].dt.year - 2021) * 12 + month - 1
    for p in ['urban', 'mountain', 'electric']:
        x[f'{p}_sin'] = x['month_sin'] * (frame['product_line'] == p)
        x[f'{p}_cos'] = x['month_cos'] * (frame['product_line'] == p)
    return pd.concat([x, pd.get_dummies(frame[CAT], dtype=float)], axis=1)

def fit_ridge(frame, alpha):
    x = design(frame)
    medians = x.median()
    x = x.fillna(medians)
    center, scale = x.mean(), x.std().replace(0, 1)
    z = np.column_stack([np.ones(len(x)), ((x - center) / scale).to_numpy(dtype=float)])
    penalty = np.eye(z.shape[1]) * alpha
    penalty[0, 0] = 0
    beta = np.linalg.solve(z.T @ z + penalty, z.T @ frame['sales_units'].to_numpy())
    return x.columns, medians, center, scale, beta

def predict_ridge(frame, model):
    columns, medians, center, scale, beta = model
    x = design(frame).reindex(columns=columns, fill_value=0).fillna(medians)
    z = np.column_stack([np.ones(len(x)), ((x - center) / scale).to_numpy(dtype=float)])
    return z @ beta

Entrenament: 648 files · Prova: 108 files
Mesos de la prova: <StringArray>
['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
Length: 6, dtype: str


### 2. Avalueu gener–juny de 2026 una vegada. No canvieu el model per millorar esta mètrica i continuar presentant-la com a prova independent.

In [2]:
# Un sol ajust, només amb entrenament (2022-2024)
model = fit_ridge(train, ALPHA)

# Una sola avaluació de la prova (gener-juny 2026)
pred = cap(test, predict_ridge(test, model))
final = score(test['sales_units'], pred)
print(f"Prova final · MAE: {final['MAE']:.2f} unitats · WAPE: {final['WAPE_%']:.1f} %")

# Només per a comparar (no per a triar): el mateix mes de l'any anterior
ref = score(test['sales_units'], cap(test, test['lag_12']))
print(f"Referència lag_12 · MAE: {ref['MAE']:.2f} unitats · WAPE: {ref['WAPE_%']:.1f} %")
print("Validació 2025 (CP3) · MAE: 11.92")

Prova final · MAE: 16.28 unitats · WAPE: 13.7 %
Referència lag_12 · MAE: 21.24 unitats · WAPE: 17.8 %
Validació 2025 (CP3) · MAE: 11.92
